![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Featureform Embeddings & Vector Search on Redis

In this recipe we register **embedding features** in [**Featureform**](https://docs.featureform.com/), materialize them to **Redis as a vector index**, and run **semantic (nearest-neighbor) search** against them with `client.nearest()`.

## Why embeddings belong in a feature store
An embedding is just a feature whose value is a vector. Treating it as a first-class Featureform feature buys you the same guarantees as any other feature: it's **defined once**, **versioned**, and **served from a low-latency online store** — here, Redis, which doubles as the vector index for similarity search. The model that produced the embedding, the source rows, and the serving index all stay linked.

## What we'll build
A tiny **semantic product search**:
1. Embed product descriptions with a sentence-transformer model.
2. Load the vectors into ClickHouse and register them as an `ff.Embedding` feature, materialized to **Redis**.
3. Embed a free-text query and ask Redis for the nearest products with `client.nearest()`.

> ℹ️ **Why the embeddings are precomputed in Python:** Featureform runs SQL transformations in the offline store, and SQL can't call a transformer model. So we compute vectors in the notebook and load them into ClickHouse. (Computing embeddings *inside* a transformation would require a Spark/Kubernetes provider.)

## The stack — all local, no Spark

- **ClickHouse** — offline store; holds the source rows and their precomputed vectors.
- **Redis** — online store **and vector index**; serves nearest-neighbor queries.
- **Featureform** coordinator — registers resources and materializes the vectors into Redis.

> ⚠️ **Needs local Docker; will not run on Colab or in CI.** You need a running Featureform coordinator (gRPC `localhost:7878`, dashboard `http://localhost` — see the [install docs](https://docs.featureform.com/deployment/quickstart-docker)) plus the two containers below.

### Start ClickHouse and Redis

In [ ]:
# NBVAL_SKIP
!docker run -d --name clickhouse -p 8123:8123 -p 9000:9000 clickhouse/clickhouse-server:latest
!docker run -d --name redis -p 6379:6379 redis:8

## Environment Setup

### Install Python Dependencies

In [ ]:
%pip install -q featureform redis clickhouse-connect sentence-transformers pandas

### Configure connections

 `host.docker.internal` is how the coordinator container reaches ClickHouse/Redis on your host (Linux: `172.17.0.1`).

In [ ]:
import os

# Featureform coordinator (gRPC)
FEATUREFORM_HOST = os.getenv("FEATUREFORM_HOST", "localhost:7878")

# Address the coordinator container uses to reach the providers.
# Mac/Windows: 'host.docker.internal'. Linux: try '172.17.0.1'.
PROVIDER_HOST = os.getenv("PROVIDER_HOST", "host.docker.internal")

# ClickHouse offline store (default user, no password)
CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", PROVIDER_HOST)
CLICKHOUSE_NATIVE_PORT = int(os.getenv("CLICKHOUSE_NATIVE_PORT", "9000"))
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "default")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "")
CLICKHOUSE_DATABASE = os.getenv("CLICKHOUSE_DATABASE", "default")

# Redis online store
REDIS_HOST = os.getenv("REDIS_HOST", PROVIDER_HOST)
REDIS_PORT = int(os.getenv("REDIS_PORT", "6379"))
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")

### Embed products and load them into ClickHouse

We embed each product description with `all-MiniLM-L6-v2` (384-dimensional vectors) and store the vectors in a ClickHouse `Array(Float32)` column. `DIMS` must match both the model and the `ff.Embedding` definition later.

In [ ]:
# NBVAL_SKIP
import clickhouse_connect
from sentence_transformers import SentenceTransformer

PRODUCTS = [
    ("p01", "Wireless noise-cancelling over-ear headphones"),
    ("p02", "Bluetooth portable speaker, waterproof"),
    ("p03", "Ergonomic mechanical keyboard with RGB backlight"),
    ("p04", "4K ultra-wide gaming monitor, 144Hz"),
    ("p05", "Stainless steel insulated water bottle"),
    ("p06", "Cast iron skillet, pre-seasoned"),
    ("p07", "Trail running shoes with grip sole"),
    ("p08", "Merino wool hiking socks, 3-pack"),
]

model = SentenceTransformer("all-MiniLM-L6-v2")
DIMS = model.get_sentence_embedding_dimension()  # 384

ids = [p[0] for p in PRODUCTS]
names = [p[1] for p in PRODUCTS]
vectors = model.encode(names).tolist()

ch = clickhouse_connect.get_client(host="localhost", port=8123,
                                   username=CLICKHOUSE_USER, password=CLICKHOUSE_PASSWORD)
ch.command("DROP TABLE IF EXISTS products")
ch.command(
    f"""
    CREATE TABLE products (
        id String,
        name String,
        embedding Array(Float32)
    ) ENGINE = MergeTree ORDER BY id
    """
)
ch.insert("products", list(zip(ids, names, vectors)),
          column_names=["id", "name", "embedding"])
print(f"loaded {len(ids)} products, {DIMS}-dim embeddings")

## Register the providers

In [ ]:
import featureform as ff

clickhouse = ff.register_clickhouse(
    name="clickhouse-quickstart",
    description="ClickHouse offline store holding product vectors",
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_NATIVE_PORT,
    user=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

redis = ff.register_redis(
    name="redis-quickstart",
    description="Redis online (inference) store",
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    db=0,
)

## Register the source and the embedding feature

We register the `products` table, then declare an `ff.Embedding` over its vector column. `vector_db=redis` tells Featureform to materialize the vectors into Redis and build a vector index there; `dims` must match the model. `@ff.entity` keys the embedding by product.

In [ ]:
products = clickhouse.register_table(
    name="products",
    variant="quickstart",
    table="products",
)

@ff.entity
class Product:
    product_embedding = ff.Embedding(
        products[["id", "embedding"]],
        dims=DIMS,
        vector_db=redis,
        variant="quickstart",
        description="Sentence-transformer embedding of the product description",
    )

## Apply

`client.apply()` registers everything and materializes the vectors into Redis, building the searchable index.

In [ ]:
# NBVAL_SKIP
client = ff.Client(host=FEATUREFORM_HOST, insecure=True)
client.apply(asynchronous=False, verbose=True)

## Semantic search from Redis

Embed a free-text query with the **same model**, then ask Redis for the nearest product embeddings. `client.nearest()` returns the entity keys (product ids) of the closest vectors — served from the Redis index.

In [ ]:
# NBVAL_SKIP
query = "something to listen to music outdoors"
query_vec = model.encode(query).tolist()

neighbors = client.nearest(
    ("product_embedding", "quickstart"),
    query_vec,
    k=3,
)

name_by_id = dict(zip(ids, names))
print(f"query: {query!r}\n")
for pid in neighbors:
    print(f"  {pid}: {name_by_id.get(pid, '?')}")

### What just happened

The nearest neighbors came back from **Redis**, not from re-scanning the source. The embedding is a normal Featureform feature — versioned and defined once — that happens to be served through a vector index. The dashboard at **http://localhost** shows it alongside every other feature, with its source lineage intact.

## Cleanup

Stop and remove the containers when you're done.

In [ ]:
# NBVAL_SKIP
!docker rm -f clickhouse redis

## Learn more

- [Featureform embeddings & vector search](https://docs.featureform.com/)
- [Featureform + Redis fraud detection recipe](./02_featureform_fraud_detection.ipynb)
- [RedisVL vector search recipes](../vector-search/) — using Redis as a vector database directly